# Evaluación Completa de Modelos SegRot

Este notebook evalúa todos los modelos entrenados (MobileNet, ResNet50, EfficientNet) en el conjunto de test, calculando:
- Métricas de segmentación (IoU, Dice)
- Métricas de clasificación (Accuracy, F1-Score)
- Matrices de confusión
- Reportes detallados por clase

In [ ]:
# Imports y configuración
from model import MobilnetSegRot, ResNet50SegRot, EfficientNetSegRot
import torch
import torch.nn as nn
import pandas as pd
from pathlib import Path
from utils import Load_Dataset
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from torchmetrics import F1Score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import datetime
import os

# Configuración
torch.manual_seed(17)
plt.style.use('default')
sns.set_palette("husl")

print("📚 Librerías cargadas exitosamente")
print(f"🖥️ Device disponible: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Funciones de métricas
def calculate_iou(predicted_mask, target_mask, epsilon=1e-6):
    """Calcula IoU entre máscaras predichas y reales"""
    predicted_mask = torch.sigmoid(predicted_mask)
    predicted_mask = (predicted_mask > 0.5).float()
    
    predicted_mask = predicted_mask.view(-1).bool()
    target_mask = target_mask.view(-1).bool()
    
    intersection = (predicted_mask & target_mask).float().sum()
    union = (predicted_mask | target_mask).float().sum()
    
    iou = (intersection + epsilon) / (union + epsilon)
    return iou.item()


def calculate_dice(predicted_mask, target_mask, epsilon=1e-6):
    """Calcula Dice coefficient entre máscaras predichas y reales"""
    predicted_mask = torch.sigmoid(predicted_mask)
    predicted_mask = (predicted_mask > 0.5).float()
    
    predicted_mask = predicted_mask.view(-1)
    target_mask = target_mask.view(-1)
    
    intersection = (predicted_mask * target_mask).sum()
    dice = (2.0 * intersection + epsilon) / (predicted_mask.sum() + target_mask.sum() + epsilon)
    return dice.item()

print("✅ Funciones de métricas definidas")

In [ ]:
# Cargar dataset
df_path = Path("../Data_df/progreso_orientacion_aument.csv").resolve()
path = Path("../Data_img/imagenes_cedula_peq").resolve()
results_dir = Path("test_results")
results_dir.mkdir(exist_ok=True)

print(f"📁 Buscando dataset en: {df_path}")
print(f"📁 Buscando imágenes en: {path}")

# Verificar archivos
if not df_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo CSV: {df_path}")
if not path.exists():
    raise FileNotFoundError(f"No se encontró el directorio de imágenes: {path}")

# Cargar datos
df = pd.read_csv(df_path).drop(columns=["Unnamed: 0"], errors="ignore")
uniques = np.unique(df['Clase'].values)
num_classes = len(uniques)
class_names = [f"Clase_{i}" for i in range(num_classes)]

print(f"📊 Dataset cargado: {len(df)} muestras, {num_classes} clases")
print(f"🏷️ Primeras 10 clases: {uniques[:10]}")

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Device: {device}")

In [ ]:
# Cargar dataset de test
dataset = Load_Dataset(df, path, batch_size=16, num_workers=4)
test_dataloader = dataset.load_test()
print(f"📦 Test DataLoader: {len(test_dataloader)} batches")
print(f"📦 Total muestras aprox: {len(test_dataloader) * 16}")

# Mostrar una muestra del dataset
for img, mask, clase in test_dataloader:
    print(f"📊 Forma de las imágenes: {img.shape}")
    print(f"📊 Forma de las máscaras: {mask.shape}")
    print(f"📊 Forma de las clases: {clase.shape}")
    print(f"📊 Rango de clases en batch: {clase.min().item()} - {clase.max().item()}")
    break

In [ ]:
# Función de evaluación de modelo
def evaluate_model(model, model_name, test_dataloader, device, class_names):
    """Evalúa un modelo específico en el conjunto de test"""
    print(f"\n🔍 EVALUANDO MODELO: {model_name}")
    print("="*50)
    
    model.eval()
    
    # Métricas para segmentación
    iou_scores = []
    dice_scores = []
    
    # Métricas para clasificación
    all_predictions = []
    all_targets = []
    
    # Métricas detalladas por sample
    detailed_results = []
    
    with torch.no_grad():
        for batch_idx, (img, mask, clase) in enumerate(tqdm(test_dataloader, desc=f"Evaluando {model_name}")):
            img = img.to(device).float()
            mask = mask.to(device).float()
            clase = clase.to(device).long()
            
            # Inferencia
            pred_mask, pred_class = model(img)
            
            # Calcular métricas por cada imagen en el batch
            for i in range(img.size(0)):
                # Métricas de segmentación
                iou = calculate_iou(pred_mask[i:i+1], mask[i:i+1])
                dice = calculate_dice(pred_mask[i:i+1], mask[i:i+1])
                
                iou_scores.append(iou)
                dice_scores.append(dice)
                
                # Métricas de clasificación
                pred_class_idx = pred_class[i].argmax().item()
                true_class_idx = clase[i].item()
                
                all_predictions.append(pred_class_idx)
                all_targets.append(true_class_idx)
                
                # Guardar resultado detallado
                detailed_results.append({
                    'model': model_name,
                    'batch': batch_idx,
                    'sample': i,
                    'iou': iou,
                    'dice': dice,
                    'pred_class': pred_class_idx,
                    'true_class': true_class_idx,
                    'correct_class': pred_class_idx == true_class_idx,
                    'pred_class_name': class_names[pred_class_idx] if pred_class_idx < len(class_names) else 'Unknown',
                    'true_class_name': class_names[true_class_idx] if true_class_idx < len(class_names) else 'Unknown'
                })
    
    # Calcular métricas agregadas
    mean_iou = np.mean(iou_scores)
    std_iou = np.std(iou_scores)
    mean_dice = np.mean(dice_scores)
    std_dice = np.std(dice_scores)
    
    # Exactitud de clasificación
    accuracy = np.mean([pred == true for pred, true in zip(all_predictions, all_targets)])
    
    # F1 Score macro
    f1_macro = F1Score(task="multiclass", num_classes=len(class_names), average="macro")
    f1_score = f1_macro(torch.tensor(all_predictions), torch.tensor(all_targets)).item()
    
    # Matriz de confusión
    cm = confusion_matrix(all_targets, all_predictions)
    
    # Reporte de clasificación
    class_report = classification_report(all_targets, all_predictions, 
                                       target_names=class_names, 
                                       output_dict=True, 
                                       zero_division=0)
    
    # Resultados agregados
    results_summary = {
        'model': model_name,
        'num_samples': len(iou_scores),
        'mean_iou': mean_iou,
        'std_iou': std_iou,
        'mean_dice': mean_dice,
        'std_dice': std_dice,
        'accuracy': accuracy,
        'f1_macro': f1_score,
        'evaluation_time': datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    
    print(f"📊 Resultados de {model_name}:")
    print(f"   IoU: {mean_iou:.4f} ± {std_iou:.4f}")
    print(f"   Dice: {mean_dice:.4f} ± {std_dice:.4f}")
    print(f"   Accuracy: {accuracy:.4f}")
    print(f"   F1-Macro: {f1_score:.4f}")
    
    return results_summary, detailed_results, cm, class_report

print("✅ Función de evaluación definida")

In [ ]:
# Función para plotear matriz de confusión
def plot_confusion_matrix(cm, class_names, model_name, save_path=None):
    """Crea y guarda la matriz de confusión"""
    plt.figure(figsize=(12, 10))
    
    # Normalizar la matriz de confusión
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Crear heatmap
    sns.heatmap(cm_normalized, 
                annot=True, 
                fmt='.2f', 
                cmap='Blues',
                xticklabels=class_names[:min(len(class_names), 20)],  # Limitar labels
                yticklabels=class_names[:min(len(class_names), 20)],
                cbar_kws={'label': 'Proporción'})
    
    plt.title(f'Matriz de Confusión - {model_name}', fontsize=16, fontweight='bold')
    plt.xlabel('Predicción', fontsize=12)
    plt.ylabel('Verdadero', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"📊 Matriz de confusión guardada: {save_path}")
    
    plt.show()

print("✅ Función de visualización definida")

In [ ]:
# Definir modelos a evaluar
models_to_evaluate = [
    ("MobilnetSegRot", MobilnetSegRot(num_classes=num_classes), "MobilnetSegRot.pth"),
    ("ResNet50SegRot", ResNet50SegRot(num_classes=num_classes), "ResNet50SegRot.pth"),
    ("EfficientNetSegRot", EfficientNetSegRot(num_classes=num_classes), "EfficientNetSegRot.pth")
]

print("🤖 Modelos definidos:")
for model_name, _, model_path in models_to_evaluate:
    exists = "✅" if os.path.isfile(model_path) else "❌"
    print(f"   {exists} {model_name}: {model_path}")

In [ ]:
# EVALUACIÓN DE TODOS LOS MODELOS
all_summaries = []
all_detailed_results = []

for model_name, model, model_path in models_to_evaluate:
    try:
        # Cargar modelo entrenado
        if os.path.isfile(model_path):
            model.load_state_dict(torch.load(model_path, map_location=device))
            print(f"✅ Modelo {model_name} cargado desde {model_path}")
        else:
            print(f"❌ No se encontró {model_path}, saltando evaluación")
            continue
        
        model.to(device)
        
        # Evaluar modelo
        summary, detailed, cm, class_report = evaluate_model(
            model, model_name, test_dataloader, device, class_names
        )
        
        # Guardar resultados
        all_summaries.append(summary)
        all_detailed_results.extend(detailed)
        
        # Crear y guardar matriz de confusión
        cm_path = results_dir / f"confusion_matrix_{model_name}.png"
        plot_confusion_matrix(cm, class_names, model_name, cm_path)
        
        # Guardar reporte de clasificación detallado
        class_report_df = pd.DataFrame(class_report).transpose()
        class_report_path = results_dir / f"classification_report_{model_name}.csv"
        class_report_df.to_csv(class_report_path)
        print(f"📊 Reporte de clasificación guardado: {class_report_path}")
        
        # Limpiar memoria
        del model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
    except Exception as e:
        print(f"❌ Error evaluando {model_name}: {str(e)}")
        continue

In [ ]:
# GUARDAR RESULTADOS FINALES
if all_summaries:
    # Resumen general
    summary_df = pd.DataFrame(all_summaries)
    summary_path = results_dir / "models_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"📊 Resumen general guardado: {summary_path}")
    
    # Mostrar resumen
    print("\n📊 RESUMEN DE RESULTADOS:")
    display(summary_df)
    
    # Resultados detallados por muestra
    detailed_df = pd.DataFrame(all_detailed_results)
    detailed_path = results_dir / "detailed_results.csv"
    detailed_df.to_csv(detailed_path, index=False)
    print(f"📊 Resultados detallados guardados: {detailed_path}")
    print(f"📊 Total de muestras evaluadas: {len(detailed_df)}")

In [ ]:
# COMPARACIÓN FINAL Y MEJORES MODELOS
if all_summaries:
    print(f"\n{'='*60}")
    print("📊 COMPARACIÓN FINAL DE MODELOS")
    print(f"{'='*60}")
    
    for _, row in summary_df.iterrows():
        print(f"🔹 {row['model']}:")
        print(f"   IoU: {row['mean_iou']:.4f} ± {row['std_iou']:.4f}")
        print(f"   Dice: {row['mean_dice']:.4f} ± {row['std_dice']:.4f}")
        print(f"   Accuracy: {row['accuracy']:.4f}")
        print(f"   F1-Macro: {row['f1_macro']:.4f}")
        print()
    
    # Mejor modelo por métrica
    best_iou = summary_df.loc[summary_df['mean_iou'].idxmax()]
    best_acc = summary_df.loc[summary_df['accuracy'].idxmax()]
    best_f1 = summary_df.loc[summary_df['f1_macro'].idxmax()]
    
    print(f"🏆 MEJORES MODELOS POR MÉTRICA:")
    print(f"   IoU: {best_iou['model']} ({best_iou['mean_iou']:.4f})")
    print(f"   Accuracy: {best_acc['model']} ({best_acc['accuracy']:.4f})")
    print(f"   F1-Macro: {best_f1['model']} ({best_f1['f1_macro']:.4f})")

print(f"\n📁 Todos los resultados guardados en: {results_dir}")
print("✅ Evaluación completa terminada!")